# pyGeoMapper — Notebook 2: Multiscale 2D CWT Terrain Complexity & Landform Scale Analysis

**Author:** Larry Syu-Heng Lai  
**License:** AGPL-3.0-or-later (`pyGeoMapper` is an independent Rust + Python geoscience toolkit and is distinct from `The-Magicians-Code/PyGeoMapper`.)

This notebook demonstrates **`terrain_analysis`** of `pyGeoMapper`, implementing a parallel Rust **2D Continuous Wavelet Transform (`2D CWT`)** with the isotropic Mexican-hat (Marr / Ricker) wavelet across spatial scales $s \in [20\text{ m}, 240\text{ m}]$:
$$\psi(x, y; s) = \frac{1}{\pi s^2}\left(1 - \frac{x^2 + y^2}{2s^2}\right)\exp\!\left(-\frac{x^2 + y^2}{2s^2}\right), \qquad \lambda(s) = \sqrt{2}\,\pi s$$

- **Methodological Citation:** Adapted from **pyTopoComplexity** (Lai, S.-H., et al., 2025, *Earth Surface Dynamics*, 13, 417–438, https://doi.org/10.5194/esurf-13-417-2025). If `pytopocomplexity` is installed in the Python environment, `terrain_analysis.run_pytopocomplexity()` can invoke it directly while defaulting seamlessly to the parallel Rust kernel (`_core.compute_multiscale_complexity_rs`).
- **Case Study 1 (Tectonic Hogbacks & Folded Bedrock):** **Sheep Mountain Anticline, Bighorn Basin, Wyoming** (`EPSG:32612`, USGS 3DEP DEM `20 m` grid) — multiscale wavelet energy isolates fine inner-canyon incisions (`s = 30 m`) vs. broad anticlinal hogback ridges (`s = 120–240 m`).
- **Case Study 2 (Catastrophic Landslide Roughness Change):** **2014 Oso (SR 530) Landslide, Washington** (`EPSG:32610`, pre-event Copernicus GLO-30 vs. post-event USGS 3DEP LiDAR DEM) — quantifying short-wavelength hummocky debris-avalanche complexity enhancement alongside 2D, 2.5D/3D, and swath cross-profile views.

### About this notebook
- **Purpose:** multiscale 2D CWT terrain complexity on Sheep Mountain (WY) and roughness change across the 2014 Oso landslide (WA).
- **Data used:** `example_data/sheep_mtn_dem_3dep.tif`, `sheep_mtn_geol_units.tif`, `sheep_mtn_units.json`, `oso_dem_pre_cop30.tif`, `oso_dem_post_3dep.tif`, `oso_naip_20130910.tif`, `oso_s2_20170826.tif`.
- **Figures written:** `image/cwt_multiscale.png` (README Sheep Mountain CWT board), `image/oso_landslide_dod.png` (README Oso pre/post/DoD), `image/sheep_mtn_cwt_3d_drape.png`.
- **How to re-run:** run all cells from `notebooks/` or the repo root. Figures are saved to `image/` with `plotting.save_figure` and are not embedded in the notebook.

In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

from pygeomapper import (
    raster_io,
    terrain_analysis,
    cross_section,
    viewer_3d,
    plotting,
)

ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)

plt.rcParams['figure.dpi'] = 120
DATA_DIR = ROOT / "example_data"
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok=True)

sheep_dem = raster_io.load_geotiff(DATA_DIR / 'sheep_mtn_dem_3dep.tif', name='sheep_dem_3dep', units='m')
sheep_geol = raster_io.load_geotiff(DATA_DIR / 'sheep_mtn_geol_units.tif', name='sheep_geol_units', units='unit_id')
oso_pre = raster_io.load_geotiff(DATA_DIR / 'oso_dem_pre_cop30.tif', name='oso_dem_pre', units='m')
oso_post = raster_io.load_geotiff(DATA_DIR / 'oso_dem_post_3dep.tif', name='oso_dem_post', units='m')

print(f'Sheep Mountain 3DEP DEM: {sheep_dem.shape}, CRS={sheep_dem.crs}, relief={np.nanmin(sheep_dem.data):.1f}..{np.nanmax(sheep_dem.data):.1f} m')
print(f'Oso Pre/Post DEMs:       {oso_post.shape}, CRS={oso_post.crs}, relief={np.nanmin(oso_post.data):.1f}..{np.nanmax(oso_post.data):.1f} m')

Sheep Mountain 3DEP DEM: (160, 160), CRS=EPSG:32612, relief=1134.0..1512.8 m
Oso Pre/Post DEMs:       (200, 200), CRS=EPSG:32610, relief=71.8..277.8 m


## 1. Synthetic Wavelength Recovery Verification (2D Mexican-Hat CWT)
Before analyzing real terrain, we verify that the Rust separable 2D Mexican-hat CWT (`compute_multiscale_complexity`) recovers the exact theoretical wavelength $\lambda_0 = 120\text{ m}$ ($s_0 = \lambda_0 / (\sqrt{2}\pi) \approx 27.01\text{ m}$) of a 2D sinusoidal ridge-and-valley surface.

In [2]:
xs_syn = np.arange(128, dtype=np.float64) * 5.0
ys_syn = np.arange(128, dtype=np.float64) * 5.0
xx, yy = np.meshgrid(xs_syn, ys_syn)
target_lambda_m = 120.0
syn_surface = 25.0 * np.sin(2.0 * np.pi * xx / target_lambda_m) + 15.0 * np.cos(2.0 * np.pi * yy / target_lambda_m)

scales_test = [12.0, 18.0, 27.0, 40.0, 60.0]
syn_cwt = terrain_analysis.compute_multiscale_complexity(
    syn_surface, scales_m=scales_test, pixel_size_m=5.0
)
core_wl = syn_cwt['dominant_wavelength_m'][25:-25, 25:-25]
median_wl = float(np.nanmedian(core_wl))
rel_err_pct = 100.0 * abs(median_wl - target_lambda_m) / target_lambda_m
print(f'Target wavelength:    {target_lambda_m:.2f} m')
print(f'Recovered wavelength: {median_wl:.2f} m (relative error = {rel_err_pct:.2f}%)')

Target wavelength:    120.00 m
Recovered wavelength: 116.84 m (relative error = 2.64%)


## 2. Multiscale 2D CWT Terrain Complexity over Sheep Mountain Anticline, WY

### 1. What This Step Does & Why We Use This Method
- Applies the parallel Rust 2D Continuous Wavelet Transform (`compute_multiscale_complexity` / `run_pytopocomplexity`, adapted from *pyTopoComplexity*, Lai et al., 2025, *ESurf*) across spatial scales $s \in \{30, 60, 120, 220\}\text{ m}$.
- Unlike standard $3\times 3$ slope or curvature (which only measure local grid-cell derivatives), 2D CWT decomposes topography by **characteristic spatial wavelength** $\lambda(s) = \sqrt{2}\pi s$.

### 2. Governing Equations & Physical Meaning
$$W(x, y; s) = (z * \psi_s)(x, y), \qquad \psi_s(x, y) = \frac{1}{\pi s^2}\left(1 - \frac{x^2 + y^2}{2s^2}\right)\exp\!\left(-\frac{x^2 + y^2}{2s^2}\right)$$
$$E(x, y; s) = \frac{|W(x, y; s)|}{s}, \qquad \lambda_{\text{dom}}(x, y) = \sqrt{2}\pi\,\arg\max_{s} E(x, y; s)$$
- **Fine scale ($s = 30\text{ m}$, $\lambda \approx 133\text{ m}$):** Isolates steep inner-canyon walls along the Bighorn River gorge and sharp flatiron/hogback crests.
- **Coarse scale ($s = 120\text{–}220\text{ m}$, $\lambda \approx 533\text{–}977\text{ m}$):** Captures the macroscopic fold ridge-and-strike-valley wavelength.


In [3]:
sheep_scales_m = [30.0, 60.0, 120.0, 220.0]
sheep_cwt = terrain_analysis.run_pytopocomplexity(sheep_dem, scales_m=sheep_scales_m)
sheep_hs = terrain_analysis.compute_hillshade(sheep_dem, azimuth_deg=315.0, altitude_deg=42.0)

# Summarize mean multiscale complexity by USGS SGMC bedrock formation
meta = json.loads((DATA_DIR / 'sheep_mtn_units.json').read_text())
print(f'Backend used: {sheep_cwt["backend"]} ({sheep_cwt["citation"]})\n')
print(f'{"Unit":6s} | {"Formation Name":32s} | {"Mean Complexity":>15s} | {"Median Wavelength (m)":>21s}')
print('-' * 84)
for u in meta['units']:
    uid = u['unit_id']
    m = sheep_geol.data == uid
    mc = float(np.nanmean(sheep_cwt['mean_complexity'].data[m]))
    mw = float(np.nanmedian(sheep_cwt['dominant_wavelength_m'].data[m]))
    print(f"{u['code']:6s} | {u['name']:32s} | {mc:15.2f} | {mw:21.1f}")

Backend used: rust_cwt2d (Lai, L. S.-H., et al. (2025). pyTopoComplexity: multiscale 2D continuous wavelet terrain complexity. Earth Surface Dynamics, 13, 417-438, https://doi.org/10.5194/esurf-13-417-2025)

Unit   | Formation Name                   | Mean Complexity | Median Wavelength (m)
------------------------------------------------------------------------------------
Mm     | Madison Limestone                |           27.24 |                 977.4
IPta   | Tensleep Sandstone & Amsden Fm   |           18.02 |                 977.4
TRPc   | Chugwater & Goose Egg Fms        |           24.08 |                 977.4
Jsg    | Sundance & Gypsum Spring Fms     |           16.31 |                 977.4
KJcm   | Cloverly & Morrison Fms          |            9.54 |                 977.4
Kmt    | Mowry & Thermopolis Shales       |            4.63 |                 977.4
Qa     | Quaternary Alluvium & Terrace    |            8.49 |                 977.4


## 3. Pre- vs. Post-Landslide Multiscale Complexity Change (2014 Oso Landslide, WA)
Next we apply `terrain_analysis.compare_complexity_change(oso_pre, oso_post, scales_m=[15, 30, 60, 120])` to quantify how the March 22, 2014 Oso landslide altered surface roughness and characteristic landform wavelength across the headscarp and hummocky debris-avalanche deposit.

In [4]:
oso_scales_m = [15.0, 30.0, 60.0, 120.0]
oso_comp = terrain_analysis.compare_complexity_change(oso_pre, oso_post, scales_m=oso_scales_m)

delta_c = oso_comp['delta_complexity'].data
ratio_c = oso_comp['complexity_ratio'].data
print(f'Oso Complexity Change (Post - Pre): min={np.nanmin(delta_c):.2f}, max=+{np.nanmax(delta_c):.2f}')
print(f'Oso Complexity Ratio  (Post / Pre): median={np.nanmedian(ratio_c):.2f}x, 95th pct={np.nanpercentile(ratio_c, 95):.2f}x')

Oso Complexity Change (Post - Pre): min=-5.73, max=+1.49
Oso Complexity Ratio  (Post / Pre): median=1.00x, 95th pct=1.04x


## 4. README Oso figure: pre/post DEM drapes + DoD (`image/oso_landslide_dod.png`)

Oso is **not** inset on the Sheep Mountain CWT board. This dedicated figure shows:
1. **Pre-slide DEM** (COP30 2011–13) draped with **USDA NAIP 2013-09-10** (α = 0.4).
2. **Post-slide DEM** (3DEP) draped with **Sentinel-2 2017-08-26** RGB (α = 0.4).
3. **DEM of Difference** (red = erosion, blue = deposit) from `surface_change.compute_dod`.

Co-event InSAR is **not** shipped: Sentinel-1 launched after 22 Mar 2014, and the package has no ALOS/UAVSAR pair spanning this pre/post DEM pair. Multiscale Δcomplexity remains available from the previous cell for notebook exploration.


In [5]:
from pygeomapper import surface_change

s2_post = raster_io.load_multiband_geotiff(
    DATA_DIR / "oso_s2_20170826.tif",
    band_names=["blue", "green", "red", "nir", "swir1", "swir2"],
)
naip = raster_io.load_multiband_geotiff(
    DATA_DIR / "oso_naip_20130910.tif", band_names=["red", "green", "blue"]
)
pre_rgb = np.stack(
    [naip.band("red").data, naip.band("green").data, naip.band("blue").data], axis=-1
).astype(np.float64)
if np.nanmax(pre_rgb) > 1.5:
    pre_rgb = pre_rgb / 255.0
pre_rgb = np.clip(pre_rgb, 0.0, 1.0)

def _s2_rgb(stack):
    rgb = np.stack([stack.band("red").data, stack.band("green").data, stack.band("blue").data], axis=-1).astype(np.float64)
    out = np.zeros_like(rgb)
    for i in range(3):
        lo, hi = np.nanpercentile(rgb[:, :, i], [2, 98])
        out[:, :, i] = np.clip((rgb[:, :, i] - lo) / max(hi - lo, 1e-9), 0.0, 1.0)
    return out

hs_pre = terrain_analysis.compute_hillshade(oso_pre)
hs_post = terrain_analysis.compute_hillshade(oso_post)
raw_dod = surface_change.compute_dod(oso_pre, oso_post)
stable = np.abs(raw_dod["raw_dod"].data) < 4.0
dod = surface_change.compute_dod(oso_pre, oso_post, detrend_stable_mask=stable)["dod"]
vol = surface_change.compute_volume_budget(dod, lod_threshold=2.0)
ext = [oso_post.bounds[0] / 1e3, oso_post.bounds[2] / 1e3, oso_post.bounds[1] / 1e3, oso_post.bounds[3] / 1e3]

fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.0), layout="constrained")
fig.suptitle("2014 Oso landslide (WA) — pre/post DEM drapes and DEM of Difference", fontsize=12.5, fontweight="bold")
axes[0].imshow(hs_pre.data, cmap="gray", extent=ext, origin="upper")
axes[0].imshow(pre_rgb, alpha=0.40, extent=ext, origin="upper")
axes[0].set_title("(a) Pre-slide DEM + NAIP 2013-09-10 (α=0.4)", fontweight="bold", fontsize=10)
axes[1].imshow(hs_post.data, cmap="gray", extent=ext, origin="upper")
axes[1].imshow(_s2_rgb(s2_post), alpha=0.40, extent=ext, origin="upper")
axes[1].set_title("(b) Post-slide DEM + Sentinel-2 2017-08-26 (α=0.4)", fontweight="bold", fontsize=10)
axes[2].imshow(hs_post.data, cmap="gray", extent=ext, origin="upper")
dod_masked = np.ma.masked_inside(dod.data, -2.0, 2.0)
im = axes[2].imshow(dod_masked, cmap=plotting.DOD_CMAP, vmin=-45, vmax=45, alpha=0.88, extent=ext, origin="upper")
axes[2].set_title(
    f"(c) DoD (red = erosion, blue = deposit)
eroded {vol['eroded_volume_m3']/1e6:.2f} Mm³ | deposited {vol['deposited_volume_m3']/1e6:.2f} Mm³",
    fontweight="bold", fontsize=10,
)
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04, label=plotting.DOD_LABEL)
for ax in axes:
    ax.set_xlabel("UTM Easting (km, Zone 10N)")
    ax.set_ylabel("UTM Northing (km)")
    plotting.use_plain_axes(ax)
out_oso = plotting.save_figure(fig, IMG_DIR / "oso_landslide_dod.png", max_width_px=1600, palette=True)
old = IMG_DIR / "oso_cwt_change.png"
if old.exists():
    old.unlink()
print("Saved:", out_oso)


Saved: image/oso_cwt_change.png


## 5. README Sheep Mountain CWT Board (`image/cwt_multiscale.png`)

### How to Read the Visual Output
1. **(a)** DEM + hillshade with Swath A–A′.
2. **(b)** Mexican-hat CWT response at $s = 30\text{ m}$ (ridges +, valleys −).
3. **(c)** Mean multiscale complexity draped on hillshade (0 → white / near-transparent; denser color = higher complexity).
4. **(d)** A–A′ elevation vs. CWT energy profile — **two columns wide** (same width as a+b), same row height as (e).
5. **(e)** Dominant wavelength $\lambda_{\mathrm{dom}}$ as a transparent color overlay on DEM hillshade (old panel-d content at the old panel-f slot).

Oso is shown in its own README figure (`image/oso_landslide_dod.png`), not as an inset here.


In [6]:
fig = plt.figure(figsize=(16.0, 10.2), layout="constrained")
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.0])
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
ax_prof = fig.add_subplot(gs[1, 0:2])
ax_lam = fig.add_subplot(gs[1, 2])
ext_s = [sheep_dem.bounds[0]/1e3, sheep_dem.bounds[2]/1e3, sheep_dem.bounds[1]/1e3, sheep_dem.bounds[3]/1e3]
p_start, p_end = (726900.0, 4942800.0), (729700.0, 4944600.0)
sheep_hs = terrain_analysis.compute_hillshade(sheep_dem)

ax_a.imshow(sheep_hs.data, cmap="gray", extent=ext_s, origin="upper")
im0 = ax_a.imshow(sheep_dem.data, cmap="terrain", alpha=0.58, extent=ext_s, origin="upper")
ax_a.plot([p_start[0]/1e3, p_end[0]/1e3], [p_start[1]/1e3, p_end[1]/1e3], "r--", lw=2.0, label="Swath A–A'")
ax_a.set_title("(a) Sheep Mountain DEM + hillshade", fontweight="bold", pad=8)
ax_a.legend(loc="lower right", fontsize=8)
fig.colorbar(im0, ax=ax_a, fraction=0.046, pad=0.04, label="Elevation (m)")

cwt_30 = sheep_cwt["cwt_stack"].band("cwt_30m").data
vmax_w = float(np.nanpercentile(np.abs(cwt_30), 98))
im1 = ax_b.imshow(cwt_30, cmap="RdBu_r", vmin=-vmax_w, vmax=vmax_w, extent=ext_s, origin="upper")
ax_b.set_title("(b) CWT response W (s = 30 m): ridges + / valleys −", fontweight="bold", pad=8)
fig.colorbar(im1, ax=ax_b, fraction=0.046, pad=0.04, label="Wavelet Amplitude (m)")

comp = np.asarray(sheep_cwt["mean_complexity"].data, dtype=np.float64)
vmax_c = float(np.nanpercentile(comp, 98))
norm_c = np.clip(comp / max(vmax_c, 1e-9), 0.0, 1.0) ** 0.85
rgba_c = plt.get_cmap("magma")(np.clip(comp / max(vmax_c, 1e-9), 0, 1))
rgba_c[..., 0] = (1 - norm_c) + norm_c * rgba_c[..., 0]
rgba_c[..., 1] = (1 - norm_c) + norm_c * rgba_c[..., 1]
rgba_c[..., 2] = (1 - norm_c) + norm_c * rgba_c[..., 2]
rgba_c[..., 3] = 0.02 + 0.78 * norm_c
ax_c.imshow(sheep_hs.data, cmap="gray", extent=ext_s, origin="upper")
ax_c.imshow(rgba_c, extent=ext_s, origin="upper")
sm_c = plt.cm.ScalarMappable(cmap="magma", norm=plt.Normalize(0.0, vmax_c)); sm_c.set_array([])
fig.colorbar(sm_c, ax=ax_c, fraction=0.046, pad=0.04, label="Mean |W(s)| / s")
ax_c.set_title("(c) Mean complexity on hillshade (s = 30–220 m)", fontweight="bold", pad=8)

mprof = cross_section.extract_multilayer_profile(
    {"Elevation": sheep_dem, "Energy_30m": sheep_cwt["energy_stack"].band("energy_30m"), "Energy_120m": sheep_cwt["energy_stack"].band("energy_120m")},
    start_xy=p_start, end_xy=p_end, n_samples=200, swath_width_m=120.0,
)
d_m = mprof["distance_m"]
ax_prof.plot(d_m, mprof["layers"]["Elevation"]["mean"], "k-", lw=2.0, label="Elevation (m)")
ax_prof.fill_between(d_m, mprof["layers"]["Elevation"]["min"], mprof["layers"]["Elevation"]["max"], color="gray", alpha=0.22)
ax_prof.set_xlabel("Distance Along Profile A–A' (m)"); ax_prof.set_ylabel("Elevation (m NAVD88)")
ax_p2 = ax_prof.twinx()
ax_p2.plot(d_m, mprof["layers"]["Energy_30m"]["mean"], color="#d95f02", lw=1.6, label="CWT Energy (s=30 m)")
ax_p2.plot(d_m, mprof["layers"]["Energy_120m"]["mean"], color="#7570b3", lw=1.8, ls="--", label="CWT Energy (s=120 m)")
ax_p2.set_ylabel("Wavelet Energy |W(s)|/s", color="#7570b3")
ax_prof.set_title("(d) Swath A–A': elevation vs. CWT energy", fontweight="bold", pad=8)
ax_prof.grid(True, ls=":", alpha=0.45)
l1, lb1 = ax_prof.get_legend_handles_labels(); l2, lb2 = ax_p2.get_legend_handles_labels()
ax_prof.legend(l1 + l2, lb1 + lb2, loc="upper right", fontsize=15, handlelength=3.4, handletextpad=0.75, borderpad=0.7, labelspacing=0.6, markerscale=1.2, framealpha=0.94)

lam = np.asarray(sheep_cwt["dominant_wavelength_m"].data, dtype=np.float64)
vmin_l, vmax_l = np.nanpercentile(lam, [2, 98])
ax_lam.imshow(sheep_hs.data, cmap="gray", extent=ext_s, origin="upper")
im_l = ax_lam.imshow(lam, cmap="viridis", vmin=vmin_l, vmax=vmax_l, alpha=0.65, extent=ext_s, origin="upper")
ax_lam.set_title("(e) Dominant wavelength λ_dom on hillshade", fontweight="bold", pad=8)
fig.colorbar(im_l, ax=ax_lam, fraction=0.046, pad=0.04, label="Wavelength λ (m)")

for a in (ax_a, ax_b, ax_c, ax_lam):
    a.set_xlabel("UTM Easting (km, Zone 12N)"); a.set_ylabel("UTM Northing (km)")
    plotting.use_plain_axes(a)
    a.set_aspect("equal", adjustable="box")
fig.suptitle("pyGeoMapper — Rust 2D Mexican-hat CWT and multiscale topographic complexity (Sheep Mountain)", fontsize=12, fontweight="bold", y=1.01)
fig.canvas.draw()
fig.set_layout_engine(None)
pos_e = ax_lam.get_position(); pos_d = ax_prof.get_position()
matched = [pos_d.x0, pos_e.y0, pos_d.width, pos_e.height]
ax_prof.set_position(matched); ax_p2.set_position(matched)
out_png = IMG_DIR / "cwt_multiscale.png"
plotting.save_figure(fig, out_png, max_width_px=1400, palette=True)
print("Saved:", out_png)


Saved: image/cwt_multiscale.png


## 6. 2.5D Draped Perspective Visualization of Sheep Mountain Multiscale Complexity (`image/sheep_mtn_cwt_3d_drape.png`)
Finally, we drape the multiscale 2D CWT complexity surface over the 3DEP DEM using `viewer_3d.plot_draped_surface_matplotlib()` to visualize how the resistant Paleozoic limestone/sandstone inner canyon and Mesozoic hogback ridges stand out in 3D perspective.

In [7]:
fig_3d, ax_3d = viewer_3d.plot_3d_terrain_drape(
    sheep_dem,
    overlay=sheep_cwt['mean_complexity'],
    cmap='magma',
    vertical_exaggeration=2.2,
    elevation_deg=36.0,
    azimuth_deg=-55.0,
    stride=1,
    title='Sheep Mountain Anticline — 3DEP surface draped with multiscale CWT complexity',
    cbar_label='Mean Multiscale CWT Complexity',
)
plotting.save_figure(fig_3d, IMG_DIR / "sheep_mtn_cwt_3d_drape.png", max_width_px=1400)

PosixPath('image/sheep_mtn_cwt_3d_drape.png')